#1. Setup e Cargas

In [2]:
import os

from google.colab import drive
drive.mount('/content/drive')

caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'

#os.listdir(caminho_pasta)

Mounted at /content/drive


In [3]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_dark"

# Carga
df_spotify = pd.read_parquet(f'{caminho_pasta}/df_spotify.parquet')
df_musicas = pd.read_parquet(f'{caminho_pasta}/df_musicas_enriquecido.parquet')
df_ranking = pd.read_parquet(f'{caminho_pasta}/df_ranking_emb.parquet')
pl_inicio_dia = pd.read_csv(f'{caminho_pasta}/playlist_inicio_dia.csv')
pl_fim_dia = pd.read_csv(f'{caminho_pasta}/playlist_fim_dia.csv')
pl_viagem = pd.read_csv(f'{caminho_pasta}/playlist_viagem_longa.csv')
cluster_meta = pd.read_csv(f'{caminho_pasta}/cluster_meta.csv')


# Paleta de cores dos clusters
paleta_cores = [
    #'#E85D5D', '#4A90D9', '#E8821A', '#1DB954' , '#A67FD0', '#FFE66D', # Originais
    #'#FF6B6B','#C8A800','#2ECC71','#00BCD4','#748FFC','#DA77DA',        # Nova Paleta
    '#DA77DA','#C8A800','#2ECC71','#00BCD4','#748FFC','#FF6B6B',        # Nova Paleta
    '#1ED760', '#F19EB5', '#7D5BA6', '#00D1FF'                         # Extras
]


caminho_mapeamento_artistas = os.path.join(caminho_pasta, 'mapeamento_artistas.csv')

if os.path.exists(caminho_mapeamento_artistas):
    df_mapeamento = pd.read_csv(caminho_mapeamento_artistas)
    with open(caminho_mapeamento_artistas) as f:
        mapeamento_artistas = dict(zip(df_mapeamento['original'], df_mapeamento['unificado']))
        print(f'Mapeamento carregado: {len(mapeamento_artistas)} entradas')
else:
    mapeamento_artistas = {}
    print('Arquivo de mapeamento não encontrado — lista vazia.')



Mapeamento carregado: 6 entradas


In [4]:
# Conferência rápida do mapeamento de cores
# Criamos o dicionário de mapeamento agora que a paleta está no topo
lista_clusters = cluster_meta.sort_values('cluster')['cluster_nome'].tolist()
cores_clusters = {
    nome: paleta_cores[i % len(paleta_cores)]
    for i, nome in enumerate(lista_clusters)
}

fig_check = go.Figure(go.Bar(
    x=list(cores_clusters.keys()),
    y=[1] * len(cores_clusters),
    marker_color=list(cores_clusters.values()),
    hovertemplate="Cluster: %{x}<extra></extra>"
))

fig_check.update_layout(
    title="Conferência: Mapeamento de Cores por Cluster",
    yaxis=dict(visible=False),
    height=300,
    showlegend=False
)

fig_check.show()

In [5]:
display(cluster_meta)

,cluster,cluster_nome,cluster_descricao
0,0,Exploração de artistas favoritos,Longevidade de artista alto - Início intencion...
1,1,Indiferença passiva,Alta retenção - Pouco replay
2,2,Tentativas que não colam,Artistas pouco ouvidos - Poucos plays - Skip alto
3,3,Longas e intencionais,Toca bastante - Retenção alta
4,4,Depende do momento,Início intencional baixo - Retenção alta - Ski...
5,5,Hits Pessoais,Muitos plays - Artistas favoritos - Retenção Alta


In [6]:
pd.set_option('display.max_columns', None)
display(df_spotify.info())
display(df_musicas.info())
display(df_ranking.info())
display(pl_inicio_dia.info())
display(pl_fim_dia.info())
display(pl_viagem.info())
display(cluster_meta.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31515 entries, 0 to 31514
Data columns (total 29 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 31515 non-null  object             
 1   ms_played                          31515 non-null  int64              
 2   master_metadata_track_name         28933 non-null  object             
 3   master_metadata_album_artist_name  28933 non-null  object             
 4   master_metadata_album_album_name   28933 non-null  object             
 5   spotify_track_uri                  28933 non-null  object             
 6   episode_name                       2581 non-null   object             
 7   episode_show_name                  2581 non-null   object             
 8   spotify_episode_uri                2581 non-null   object             
 9   audiobook_title                    0 non-null     

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24040 entries, 0 to 24039
Data columns (total 39 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 24040 non-null  object             
 1   ms_played                          24040 non-null  int64              
 2   master_metadata_track_name         24040 non-null  object             
 3   master_metadata_album_artist_name  24040 non-null  object             
 4   master_metadata_album_album_name   24040 non-null  object             
 5   spotify_track_uri                  24040 non-null  object             
 6   reason_start                       24040 non-null  object             
 7   reason_end                         24040 non-null  object             
 8   shuffle                            24040 non-null  bool               
 9   skipped                            24040 non-null 

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4212 entries, 0 to 4211
Data columns (total 83 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   artista_unificado            4212 non-null   object 
 1   musica_unificada             4212 non-null   object 
 2   total_plays                  4212 non-null   int64  
 3   total_minutos                4212 non-null   float64
 4   retencao_media               4212 non-null   float64
 5   escuta_completa_rate         4212 non-null   float64
 6   skip_rate                    4212 non-null   float64
 7   inicio_intencional_rate      4212 non-null   float64
 8   duracao_estimada_ms          4212 non-null   float64
 9   fonte_duracao                4212 non-null   object 
 10  data_release                 4212 non-null   object 
 11  explicit                     4212 non-null   bool   
 12  rating_lealdade_artista      4212 non-null   float64
 13  longevidade_dias_a

...

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   100 non-null    int64  
 1   musica_unificada             100 non-null    object 
 2   artista_unificado            100 non-null    object 
 3   cluster                      100 non-null    int64  
 4   freq_contexto                100 non-null    int64  
 5   retencao_media               100 non-null    float64
 6   total_plays                  100 non-null    int64  
 7   duracao_estimada_ms          100 non-null    float64
 8   spotify_track_uri_dominante  100 non-null    object 
dtypes: float64(2), int64(4), object(3)
memory usage: 7.2+ KB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   cluster            6 non-null      int64 
 1   cluster_nome       6 non-null      object
 2   cluster_descricao  6 non-null      object
dtypes: int64(1), object(2)
memory usage: 276.0+ bytes


None

In [7]:


# Garantir datetime com timezone removido (facilita agrupamentos)
df_spotify["data_finalizacao"] = pd.to_datetime(
    df_spotify["data_finalizacao"], utc=True
).dt.tz_localize(None)

# Flag: é música ou podcast?
df_spotify["tipo"] = np.where(
    df_spotify["spotify_track_uri"].notna(), "Música",
    np.where(df_spotify["spotify_episode_uri"].notna(), "Podcast", "Outro")
)

# Anos disponíveis (para dropdowns)
ANOS = sorted(df_spotify["ano"].unique())
ANO_COMPLETO_MIN = ANOS[0]
ANO_COMPLETO_MAX = ANOS[-1]


#2.1 - Gráficos por ano

In [8]:
ANOS = sorted(df_spotify["ano"].unique())

resumo_anos = {}

def truncar_label(label, max_chars=45):
    return label if len(label) <= max_chars else label[:max_chars] + "..."

for ano in ANOS:
    df_ano = df_spotify[df_spotify["ano"] == ano]
    df_ano_musicas = df_ano[df_ano["tipo"] == "Música"]
    df_ano_podcasts = df_ano[df_ano["tipo"] == "Podcast"]

    # Top 10 artistas por minutos
    top_artistas = (
        df_ano_musicas
        .groupby("artista_unificado")["minutos_tocados"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={
            "artista_unificado": "artista",
            "minutos_tocados": "minutos"
        })
    )


      # Top 10 músicas por minutos
    top_musicas = (
        df_ano_musicas[df_ano_musicas["minutos_tocados"] >= 0.5]
        .groupby(["musica_unificada", "artista_unificado"])["minutos_tocados"]  # <- troca ms_played
        .sum()                                                                   # <- troca count por sum
        .sort_values(ascending=False)
        .head(10)
        .sort_values(ascending=True)
        .reset_index()
        .rename(columns={
            "musica_unificada": "musica",
            "artista_unificado": "artista",
            "minutos_tocados": "minutos"       # <- atualiza o nome
        })
    )

    # Label legível para o eixo
    #top_musicas["label"] = top_musicas["musica"] + " — " + top_musicas["artista"]
    top_musicas["label"] = (top_musicas["musica"] + " — " + top_musicas["artista"]).apply(truncar_label)

    # KPIs
    kpis = {
        "minutos_totais": round(df_ano["minutos_tocados"].sum()),
        "musicas_unicas": df_ano_musicas[
            df_ano_musicas["minutos_tocados"] >= 0.5
        ]["musica_unificada"].nunique(),
        "artistas_unicos": df_ano_musicas["artista_unificado"].nunique(),
        "episodios_podcast": df_ano_podcasts["episode_name"].nunique(),
    }

    resumo_anos[ano] = {
        "top_artistas": top_artistas,
        "top_musicas": top_musicas,
        "kpis": kpis,
    }

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Layout: 2 linhas, 5 colunas
# Col 1-3: gráficos de barra | Col 4-5: KPIs
fig_retro = make_subplots(
    rows=2, cols=5,
    column_widths=[0.22, 0.22, 0.22, 0.17, 0.17],
    specs=[
        [{"colspan": 3}, None, None, {"type": "indicator"}, {"type": "indicator"}],
        [{"colspan": 3}, None, None, {"type": "indicator"}, {"type": "indicator"}],
    ],
    subplot_titles=[""] * 6,  # preenchemos via anotações depois
    vertical_spacing=0.12,
)

TRACES_POR_ANO = 6  # 2 barras + 4 indicators
traces_visiveis = []  # controle de visibilidade por ano

for i, ano in enumerate(ANOS):
    d = resumo_anos[ano]
    visivel = (ano == ANOS[0])  # só o primeiro ano visível no início
    kpis = d["kpis"]

    # ── Trace 1: Top artistas ──
    fig_retro.add_trace(
        go.Bar(
            x=d["top_artistas"]["minutos"],
            y=d["top_artistas"]["artista"],
            orientation="h",
            name=f"Artistas {ano}",
            marker_color="#1DB954",  # verde Spotify
            visible=visivel,
            hovertemplate="%{y}<br>%{x:.0f} min<extra></extra>",
        ),
        row=1, col=1,
    )

    # ── Trace 2: Top músicas ──
    fig_retro.add_trace(
        go.Bar(
            #x=d["top_musicas"]["plays"],
            x=d["top_musicas"]["minutos"],
            y=d["top_musicas"]["label"],
            orientation="h",
            name=f"Músicas {ano}",
            marker_color="#1ED760",
            visible=visivel,
            #hovertemplate="%{y}<br>%{x} plays<extra></extra>",  # por play
            hovertemplate="%{y}<br>%{x:.0f} min<extra></extra>", # por minutos
        ),
        row=2, col=1,
    )

    # ── Traces 3-6: KPIs ──

    kpi_cfg = [
        ("minutos_totais", "Minutos ouvidos", 1, 4),
        ("musicas_unicas",  "Músicas únicas",  1, 5),
        ("artistas_unicos", "Artistas únicos", 2, 4),
        ("episodios_podcast", "Episódios de podcast", 2, 5),
    ]

    for chave, titulo, r, c in kpi_cfg:
        fig_retro.add_trace(
            go.Indicator(
                mode="number",
                value=kpis[chave],
                title={"text": titulo, "font": {"size": 13}},
                number={"font": {"size": 36}},
                visible=visivel,
            ),
            row=r, col=c,
        )

    traces_visiveis.append([False] * TRACES_POR_ANO * len(ANOS))

# Montar máscara de visibilidade por ano
visibility_masks = []
for i in range(len(ANOS)):
    mask = [False] * (TRACES_POR_ANO * len(ANOS))
    start = i * TRACES_POR_ANO
    for j in range(TRACES_POR_ANO):
        mask[start + j] = True
    visibility_masks.append(mask)

# Botões do dropdown
botoes = []
for i, ano in enumerate(ANOS):
    #sufixo = " ★ parcial" if ano == 2026 else ""
    sufixo = ""
    botoes.append(dict(
        label=f"{ano}{sufixo}",
        method="update",
        args=[
            {"visible": visibility_masks[i]},
            {"title": {"text": f"Retrospectiva {ano}{sufixo}"}},
        ],
    ))

fig_retro.update_layout(
    title=f"Retrospectiva {ANOS[0]}",
    updatemenus=[dict(
        buttons=botoes,
        direction="down",
        x=0.01,
        xanchor="left",
        y=1.15,
        yanchor="top",
        bgcolor="#222",
        font=dict(color="white"),
    )],
    height=700,
    showlegend=False,
    yaxis=dict(autorange="reversed"),   # maior no topo — artistas
    yaxis3=dict(autorange="reversed"),  # maior no topo — músicas
    margin=dict(l=200),                 # espaço para nomes longos
)

fig_retro.show()

In [10]:
minutos_tipo_ano = (
    df_spotify
    .groupby(["ano", "tipo"])["minutos_tocados"]
    .sum()
    .reset_index()
)

# Pivotar para ter uma coluna por tipo
minutos_pivot = minutos_tipo_ano.pivot(
    index="ano", columns="tipo", values="minutos_tocados"
).fillna(0).reset_index()

# Garantir que as colunas existem mesmo que um tipo não apareça em algum ano
for col in ["Música", "Podcast", "Outro"]:
    if col not in minutos_pivot.columns:
        minutos_pivot[col] = 0

# Total para anotação
minutos_pivot["total"] = minutos_pivot["Música"] + minutos_pivot["Podcast"] + minutos_pivot["Outro"]

In [11]:
fig_minutos = go.Figure()

fig_minutos.add_trace(go.Bar(
    x=minutos_pivot["ano"],
    y=minutos_pivot["Música"],
    name="Música",
    marker_color="#1DB954",
    hovertemplate="Música<br>%{x}<br>%{y:,.0f} min<extra></extra>",
))

fig_minutos.add_trace(go.Bar(
    x=minutos_pivot["ano"],
    y=minutos_pivot["Podcast"],
    name="Podcast",
    marker_color="#5E17EB",
    hovertemplate="Podcast<br>%{x}<br>%{y:,.0f} min<extra></extra>",
))

# Anotação do total no topo de cada barra
for _, row in minutos_pivot.iterrows():
    fig_minutos.add_annotation(
        x=row["ano"],
        y=row["total"],
        text=f"{row['total']:,.0f}",
        showarrow=False,
        yshift=10,
        font=dict(size=11, color="white"),
    )

fig_minutos.update_layout(
    barmode="stack",
    title="Minutos ouvidos por ano — Música vs Podcast",
    xaxis=dict(
        tickmode="array",
        tickvals=minutos_pivot["ano"].tolist(),
        title="Ano",
    ),
    yaxis=dict(title="Minutos"),
    legend=dict(orientation="h", y=1.1, x=0.5, xanchor="center"),
    height=450,
)

fig_minutos.show()

In [12]:
# Top 10 artistas por ano — em minutos, com filtro de 30s
ranking_artistas = []

for ano in ANOS:
    top = (
        df_spotify[
            (df_spotify["ano"] == ano) &
            (df_spotify["tipo"] == "Música") &
            (df_spotify["minutos_tocados"] >= 0.5)
        ]
        .groupby("artista_unificado")["minutos_tocados"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    top["posicao"] = range(1, len(top) + 1)
    top["ano"] = ano
    ranking_artistas.append(top)

df_bump = pd.concat(ranking_artistas, ignore_index=True)

# Todos os artistas que aparecem em pelo menos um ano
artistas_bump = df_bump["artista_unificado"].unique()

print(f"Total de artistas que entraram no top 10 em algum momento: {len(artistas_bump)}")

Total de artistas que entraram no top 10 em algum momento: 42


In [13]:
ranking_artistas_5 = []

for ano in ANOS:
    top = (
        df_spotify[
            (df_spotify["ano"] == ano) &
            (df_spotify["tipo"] == "Música") &
            (df_spotify["minutos_tocados"] >= 0.5)
        ]
        .groupby("artista_unificado")["minutos_tocados"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )
    top["posicao"] = range(1, len(top) + 1)
    top["ano"] = ano
    ranking_artistas_5.append(top)

df_bump_5 = pd.concat(ranking_artistas_5, ignore_index=True)

aparicoes_5 = df_bump_5.groupby("artista_unificado")["ano"].count().sort_values(ascending=False)
print(f"Total de artistas únicos: {aparicoes_5.count()}")
print()
print(aparicoes_5.to_string())

Total de artistas únicos: 21

artista_unificado
Engenheiros e Humberto Gessinger    8
Bad Religion                        6
Black Pantera                       3
Liniker                             3
ANGRA                               2
Johnny Hooker                       2
Arnaldo Antunes                     2
Bush                                1
Cidadão Quem                        1
Garotos Podres                      1
Legião Urbana                       1
Lulu Santos                         1
Marisa Monte                        1
Novos Baianos                       1
Pink Floyd                          1
Pitty                               1
Plebe Rude                          1
Raul Seixas                         1
The Rolling Stones                  1
The Strokes                         1
Weezer                              1


In [16]:
# Separar recorrentes (2+ anos) dos ocasionais
recorrentes = set(aparicoes_5[aparicoes_5 >= 2].index)

#Paleta para recorrentes — uma cor por artista
cores_recorrentes = [
    "#1DB954", "#FF6B6B", "#4ECDC4", "#FFE66D",
    "#A8E6CF", "#FF8B94", "#B4A7D6"
]


# cores_recorrentes = [
#     "#1DB954", "#FF6B6B", "#4ECDC4", "#FFE66D",
#     "#A8E6CF", "#E82729", "#B4A7D6"
# ]
mapa_cores = {
    artista: cores_recorrentes[i % len(cores_recorrentes)]
    for i, artista in enumerate(sorted(recorrentes))
}

fig_bump = go.Figure()

for artista in df_bump_5["artista_unificado"].unique():
    df_artista = df_bump_5[df_bump_5["artista_unificado"] == artista].sort_values("ano")

    eh_recorrente = artista in recorrentes

    fig_bump.add_trace(go.Scatter(
        x=df_artista["ano"],
        y=df_artista["posicao"],
        mode="lines+markers+text",
        cliponaxis=False,
        name=artista,
        line=dict(
            color=mapa_cores.get(artista, "#444444"),
            width=3 if eh_recorrente else 1,
        ),
        marker=dict(
            size=10 if eh_recorrente else 6,
            color=mapa_cores.get(artista, "#444444"),
        ),
        #text=[artista if ano == df_artista["ano"].max() else ""
        #      for ano in df_artista["ano"]] if eh_recorrente else [],
        text=[artista if ano == df_artista["ano"].max() else ""
              for ano in df_artista["ano"]],
        #textposition="middle right",
        textposition="top right",
        textfont=dict(
            size=11 if eh_recorrente else 9,
            color=mapa_cores.get(artista, "#666666"),
        ),
        opacity=1.0 if eh_recorrente else 0.8,
        hovertemplate=f"<b>{artista}</b><br>%{{x}}<br>Posição: %{{y}}º<extra></extra>",
    ))

fig_bump.update_layout(
    title="Evolução dos artistas favoritos — Top 5 por ano",
    xaxis=dict(
        tickmode="array",
        tickvals=ANOS,
        title="Ano",
    ),
    yaxis=dict(
        autorange="reversed",  # posição 1 no topo
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5],
        ticktext=["1º", "2º", "3º", "4º", "5º"],
        title="Posição",
    ),
    height=500,
    #width=1600, # Added width to increase the chart size
    showlegend=False,  # os labels no gráfico já identificam
    margin=dict(r=350),  # espaço para os nomes à direita
)

fig_bump.show()

#2.2 - Gráficos por mês, dia da semana e horário





In [17]:
heatmap_mes = (
    df_spotify
    .groupby(["ano", "mes"])["minutos_tocados"]
    .sum()
    .reset_index()
)

heatmap_mes_pivot = heatmap_mes.pivot(
    index="ano", columns="mes", values="minutos_tocados"
).fillna(0)

nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
               "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

fig_mes = go.Figure(go.Heatmap(
    z=heatmap_mes_pivot.values,
    x=nomes_meses,
    y=heatmap_mes_pivot.index.tolist(),
    colorscale="Greens",
    hovertemplate="<b>%{y} — %{x}</b><br>%{z:,.0f} min<extra></extra>",
    colorbar=dict(title="Minutos"),
))

fig_mes.update_layout(
    title="Minutos ouvidos por ano e mês",
    xaxis=dict(title="Mês"),
    yaxis=dict(
        title="Ano",
        autorange="reversed",
        tickmode="array",
        tickvals=heatmap_mes_pivot.index.tolist(),
    ),
    height=380,
)

fig_mes.show()

In [18]:
ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nomes_dias = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

heatmap_data = (
    df_spotify
    .groupby(["dia_semana", "hora"])["minutos_tocados"]
    .sum()
    .reset_index()
)

# Pivotar para matriz hora x dia
heatmap_pivot = heatmap_data.pivot(
    index="dia_semana", columns="hora", values="minutos_tocados"
).fillna(0)

# Reordenar dias
heatmap_pivot = heatmap_pivot.reindex(ordem_dias)

In [19]:
fig_heatmap = go.Figure(go.Heatmap(
    z=heatmap_pivot.values,
    x=list(range(24)),
    y=nomes_dias,
    colorscale="Greens",
    hovertemplate="<b>%{y}</b><br>%{x}h<br>%{z:.0f} min<extra></extra>",
    colorbar=dict(title="Minutos"),
))

fig_heatmap.update_layout(
    title="Hábitos de escuta — hora × dia da semana",
    xaxis=dict(
        title="Hora do dia",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h}h" for h in range(24)],
    ),
    yaxis=dict(
        title="",
        autorange="reversed",  # Segunda no topo
    ),
    height=380,
)

fig_heatmap.show()

In [20]:
ordem_periodos = ["madrugada", "noite", "tarde", "manha"]

periodo_data = (
    df_spotify
    .groupby("periodo_dia")["minutos_tocados"]
    .sum()
    .reset_index()
    .rename(columns={"periodo_dia": "periodo", "minutos_tocados": "minutos"})
)

# Aplicar ordem cronológica — invertida porque autorange reversed
periodo_data["periodo"] = pd.Categorical(
    periodo_data["periodo"], categories=ordem_periodos, ordered=True
)
periodo_data = periodo_data.sort_values("periodo")

In [21]:
fig_periodo = go.Figure(go.Bar(
    x=periodo_data["minutos"],
    y=periodo_data["periodo"],
    orientation="h",
    marker_color="#1DB954",
    hovertemplate="%{y}<br>%{x:,.0f} min<extra></extra>",
))

fig_periodo.update_layout(
    title="Minutos ouvidos por período do dia",
    xaxis=dict(title="Minutos"),
yaxis=dict(
    ticktext=["Madrugada", "Noite", "Tarde", "Manhã"],
    tickvals=["madrugada", "noite", "tarde", "manha"]),
    height=350,
)

fig_periodo.show()

#2.3 - Gráficos por Sessão

In [22]:
sessoes = (
    df_musicas
    .groupby(["dia_ajustado", "sessao_id"])
    .agg(
        musicas=("musica_unificada", "count"),
        minutos=("minutos_tocados", "sum"),
        hora_inicio=("hora", "min"),
        ano=("ano", "first"),
    )
    .reset_index()
)

print(f"Total de sessões: {len(sessoes)}")
print(f"\nMúsicas por sessão:")
print(sessoes["musicas"].describe())
print(f"\nMinutos por sessão:")
print(sessoes["minutos"].describe())

Total de sessões: 2371

Músicas por sessão:
count    2371.000000
mean       10.139182
std        13.083795
min         1.000000
25%         1.000000
50%         5.000000
75%        15.000000
max       137.000000
Name: musicas, dtype: float64

Minutos por sessão:
count    2371.000000
mean       23.939552
std        31.096030
min         0.000000
25%         2.254033
50%        10.240350
75%        35.489333
max       286.317683
Name: minutos, dtype: float64


In [23]:
print("Sessões por tamanho:")
bins = [0, 1, 5, 10, 20, 50, 100, 200]
labels = ["1", "2-5", "6-10", "11-20", "21-50", "51-100", "100+"]
sessoes["faixa_musicas"] = pd.cut(sessoes["musicas"], bins=bins, labels=labels)
print(sessoes["faixa_musicas"].value_counts().sort_index())

Sessões por tamanho:
faixa_musicas
1         791
2-5       449
6-10      315
11-20     440
21-50     338
51-100     35
100+        3
Name: count, dtype: int64


In [24]:
faixas_ordem = ["1", "2-5", "6-10", "11-20", "21-50", "51-100", "100+"]

faixas_data = (
    sessoes["faixa_musicas"]
    .value_counts()
    .reindex(faixas_ordem)
    .reset_index()
    .rename(columns={"faixa_musicas": "faixa", "count": "sessoes"})
)

fig_sessoes = go.Figure(go.Bar(
    x=faixas_data["faixa"],
    y=faixas_data["sessoes"],
    marker_color="#1DB954",
    text=faixas_data["sessoes"],
    textposition="outside",
    textfont=dict(size=12, color="white"),
    hovertemplate="Faixa: %{x} músicas<br>Sessões: %{y}<extra></extra>",
))

fig_sessoes.update_layout(
    title="Distribuição do tamanho das sessões",
    xaxis=dict(title="Músicas por sessão", categoryorder="array", categoryarray=faixas_ordem),
    yaxis=dict(
        title="Número de sessões",
        range=[0, faixas_data["sessoes"].max() * 1.15],
    ),
    height=380,
)

fig_sessoes.show()

In [25]:
sessoes_ano = (
    sessoes
    .groupby("ano")
    .agg(
        minutos_medio=("minutos", "mean"),
        musicas_medio=("musicas", "mean"),
        total_sessoes=("sessao_id", "count"),
    )
    .reset_index()
    .round(1)
)

print(sessoes_ano)

    ano  minutos_medio  musicas_medio  total_sessoes
0  2019           23.0           10.7            274
1  2020           10.0            5.3            167
2  2021           17.8           10.4            214
3  2022           20.2            9.7            286
4  2023           20.4            9.4            123
5  2024           30.4           11.2            448
6  2025           28.9           11.3            662
7  2026           20.3            7.9            197


In [26]:
fig_sessoes_ano = make_subplots(specs=[[{"secondary_y": True}]])

fig_sessoes_ano.add_trace(
    go.Bar(
        x=sessoes_ano["ano"],
        y=sessoes_ano["minutos_medio"],
        name="Minutos médios",
        marker_color="#1DB954",
        text=sessoes_ano["minutos_medio"],
        textposition="outside",
        textfont=dict(size=11, color="white"),
        hovertemplate="%{x}<br>%{y} min médios<extra></extra>",
    ),
    secondary_y=False,
)

fig_sessoes_ano.add_trace(
    go.Scatter(
        x=sessoes_ano["ano"],
        y=sessoes_ano["musicas_medio"],
        name="Músicas médias",
        mode="lines+markers",
        line=dict(color="#FFE66D", width=2),
        marker=dict(size=8),
        hovertemplate="%{x}<br>%{y} músicas médias<extra></extra>",
    ),
    secondary_y=True,
)

fig_sessoes_ano.update_layout(
    title="Duração média das sessões por ano",
    xaxis=dict(
        tickmode="array",
        tickvals=sessoes_ano["ano"].tolist(),
    ),
    yaxis=dict(
        title="Minutos médios",
        range=[0, sessoes_ano["minutos_medio"].max() * 1.2],
    ),
    yaxis2=dict(
        title="Músicas médias",
        range=[0, sessoes_ano["musicas_medio"].max() * 1.2],
        overlaying="y",
        side="right",
    ),
    height=420,
    legend=dict(orientation="h", y=1.1, x=0.5, xanchor="center"),
)

fig_sessoes_ano.show()

In [27]:
hora_inicio_data = (
    sessoes
    .groupby("hora_inicio")["sessao_id"]
    .count()
    .reset_index()
    .rename(columns={"hora_inicio": "hora", "sessao_id": "sessoes"})
)

print(hora_inicio_data)

    hora  sessoes
0      0      144
1      1       73
2      2       52
3      3       66
4      4       57
5      5       25
6      6       53
7      7       24
8      8       10
9      9       10
10    10        9
11    11       22
12    12       82
13    13      104
14    14      127
15    15      224
16    16      258
17    17      214
18    18      189
19    19      154
20    20      133
21    21      108
22    22      129
23    23      104


In [28]:
fig_hora_inicio = go.Figure(go.Bar(
    x=hora_inicio_data["hora"],
    y=hora_inicio_data["sessoes"],
    marker_color="#1DB954",
    text=hora_inicio_data["sessoes"],
    textposition="outside",
    textfont=dict(size=10, color="white"),
    hovertemplate="%{x}h<br>%{y} sessões iniciadas<extra></extra>",
))

fig_hora_inicio.update_layout(
    title="Horário de início das sessões",
    xaxis=dict(
        title="Hora do dia",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h}" for h in range(24)],
    ),
    yaxis=dict(
        title="Sessões iniciadas",
        range=[0, hora_inicio_data["sessoes"].max() * 1.2],
    ),
    height=400,
)

fig_hora_inicio.show()

#2.4.Gráficos por data de lançamento

In [29]:


# Verificar nulos primeiro
print(f"Nulos em data_release: {df_musicas['data_release'].isna().sum()}")
print(f"Total de músicas únicas: {df_musicas['spotify_track_uri'].nunique()}")

# Trabalhar só com quem tem data
df_com_data = df_musicas[df_musicas['data_release'].notna()].copy()

print("\n=== Distribuição de granularidade ===")
print(df_com_data['data_release'].str.len().value_counts().sort_index())

print("\n=== Exemplos por tamanho ===")
for tamanho, label in [(4, 'Só ano'), (7, 'Ano-mês'), (10, 'Data completa')]:
    mask = df_com_data['data_release'].str.len() == tamanho
    n = mask.sum()
    exemplo = df_com_data[mask]['data_release'].iloc[0] if n > 0 else 'nenhum'
    print(f"{label} ({tamanho} chars): {n} músicas — ex: {exemplo}")

print("\n=== Data release >= 2019 com data completa ===")
mask_recente = df_com_data['data_release'].str[:4].astype(int) >= 2019
mask_completa = df_com_data['data_release'].str.len() == 10
print(f"Lançadas >= 2019: {mask_recente.sum()}")
print(f"Dessas, com data completa: {(mask_recente & mask_completa).sum()}")

Nulos em data_release: 0
Total de músicas únicas: 5374

=== Distribuição de granularidade ===
data_release
4      3994
10    20046
Name: count, dtype: int64

=== Exemplos por tamanho ===
Só ano (4 chars): 3994 músicas — ex: 1993
Ano-mês (7 chars): 0 músicas — ex: nenhum
Data completa (10 chars): 20046 músicas — ex: 1972-06-06

=== Data release >= 2019 com data completa ===
Lançadas >= 2019: 5051
Dessas, com data completa: 5049


In [30]:
# Extrair ano de lançamento (funciona pros dois formatos: '1993' e '1972-06-06')
df_com_data['ano_lancamento'] = df_com_data['data_release'].str[:4].astype(int)

# Agregar minutos por ano de lançamento (músicas únicas — evita contar a mesma música várias vezes)
# Usamos df_musicas pois cada linha é uma execução
df_grafico1 = (
    df_com_data
    .groupby('ano_lancamento')['minutos_tocados']
    .sum()
    .reset_index()
    .sort_values('ano_lancamento')
)

# Cobertura
total_min = df_musicas['minutos_tocados'].sum()
min_com_data = df_com_data['minutos_tocados'].sum()
cobertura = min_com_data / total_min * 100

print(f"Cobertura: {cobertura:.1f}% dos minutos totais")
print(f"Anos cobertos: {df_grafico1['ano_lancamento'].min()} → {df_grafico1['ano_lancamento'].max()}")
print(df_grafico1.tail(10))

Cobertura: 100.0% dos minutos totais
Anos cobertos: 1961 → 2026
    ano_lancamento  minutos_tocados
56            2017       824.712800
57            2018      1175.062267
58            2019      2158.540633
59            2020       942.959050
60            2021      1634.986850
61            2022      1884.861733
62            2023      1447.265850
63            2024      4138.856983
64            2025      1379.731200
65            2026        65.069817


In [31]:
# === Setup Seção 5.5 ===

# Top 3 anos
top3_anos = df_grafico1.nlargest(3, 'minutos_tocados')

# Pegar top 2 (os que vamos anotar)
ano_pico1 = top3_anos.iloc[1]['ano_lancamento']  # 1996
min_pico1 = top3_anos.iloc[1]['minutos_tocados']

ano_pico2 = top3_anos.iloc[0]['ano_lancamento']  # 2024
min_pico2 = top3_anos.iloc[0]['minutos_tocados']

# Top 3 artistas de cada pico
def top3_artistas_ano(ano):
    return (
        df_com_data[df_com_data['ano_lancamento'] == ano]
        .groupby('artista_unificado')['minutos_tocados']
        .sum()
        .nlargest(3)
        .index.tolist()
    )

artistas_pico1 = top3_artistas_ano(ano_pico1)
artistas_pico2 = top3_artistas_ano(ano_pico2)

# Formatar label: "Artista1\nArtista2 · Artista3"
def formatar_label(ano, artistas):
    return f'<b>{int(ano)}</b><br>{artistas[0]}<br>{artistas[1]} · {artistas[2]}'

label_pico1 = formatar_label(ano_pico1, artistas_pico1)
label_pico2 = formatar_label(ano_pico2, artistas_pico2)

# Limite do eixo Y com folga de 10%
ymax = df_grafico1['minutos_tocados'].max() * 1.1

print(f"Pico 1: {ano_pico1} — {min_pico1:.0f} min — {artistas_pico1}")
print(f"Pico 2: {ano_pico2} — {min_pico2:.0f} min — {artistas_pico2}")

Pico 1: 1996.0 — 3151 min — ['Bad Religion', 'ANGRA', 'Legião Urbana']
Pico 2: 2024.0 — 4139 min — ['Liniker', 'Black Pantera', 'Nando Reis']


In [32]:


# Média móvel de 5 anos
df_grafico1['media_movel'] = df_grafico1['minutos_tocados'].rolling(window=5, center=True).mean()

fig_lancamento = go.Figure()

# Barras
fig_lancamento.add_trace(go.Bar(
    x=df_grafico1['ano_lancamento'],
    y=df_grafico1['minutos_tocados'],
    marker_color='#1DB954',
    opacity=0.7,
    name='Minutos por ano',
    hovertemplate='<b>%{x}</b><br>%{y:.0f} minutos<extra></extra>',
))

# Linha de média móvel
fig_lancamento.add_trace(go.Scatter(
    x=df_grafico1['ano_lancamento'],
    y=df_grafico1['media_movel'],
    mode='lines',
    line=dict(color='#FFFFFF', width=2, dash='dot'),
    name='Média móvel (5 anos)',
    hovertemplate='<b>%{x}</b><br>Média: %{y:.0f} min<extra></extra>',
))

# Anotação pico 1
fig_lancamento.add_annotation(
    x=ano_pico1, y=min_pico1,
    text=label_pico1,
    showarrow=True,
    arrowhead=2,
    arrowcolor='#AAAAAA',
    ax=0, ay=-70,
    font=dict(size=11),
    align='center',
)

# Anotação pico 2
fig_lancamento.add_annotation(
    x=ano_pico2, y=min_pico2,
    text=label_pico2,
    showarrow=True,
    arrowhead=2,
    arrowcolor='#AAAAAA',
    ax=0, ay=-70,
    font=dict(size=11),
    align='center',
)

fig_lancamento.update_layout(
    title=dict(
        text=f'Minutos ouvidos por ano de lançamento<br><sup>Cobertura: {cobertura:.1f}% dos minutos totais</sup>',
        x=0.5
    ),
    xaxis_title='Ano de lançamento',
    yaxis_title='Minutos ouvidos',
    xaxis=dict(dtick=5),
    yaxis=dict(range=[0, ymax]),
    bargap=0.2,
    legend=dict(orientation='h', y=-0.15),
)

fig_lancamento.show()

In [33]:
# Data de lançamento do álbum = data mínima entre as faixas
data_album = (
    df_com_data[df_com_data['ano_lancamento'] >= 2019]
    .groupby('master_metadata_album_album_name')['data_release']
    .min()
    .reset_index()
    .rename(columns={'data_release': 'data_lancamento_album'})
)

# Join com execuções
df_empolgacao = df_com_data[df_com_data['ano_lancamento'] >= 2019].merge(
    data_album, on='master_metadata_album_album_name', how='left'
)

# Converter para datetime
df_empolgacao['data_lancamento_album'] = pd.to_datetime(
    df_empolgacao['data_lancamento_album'], format='mixed'
)
df_empolgacao['data_finalizacao_dt'] = pd.to_datetime(df_empolgacao['data_finalizacao'], utc=True).dt.tz_localize(None)
df_empolgacao['data_lancamento_album'] = df_empolgacao['data_lancamento_album'].dt.tz_localize(None)

# Dias desde o lançamento
df_empolgacao['dias_apos_lancamento'] = (
    df_empolgacao['data_finalizacao_dt'] - df_empolgacao['data_lancamento_album']
).dt.days

# Filtrar janela de 30 dias — só execuções após o lançamento
df_janela = df_empolgacao[
    (df_empolgacao['dias_apos_lancamento'] >= 0) &
    (df_empolgacao['dias_apos_lancamento'] <= 30)
].copy()

# Agregar por artista
df_grafico2 = (
    df_janela
    .groupby('artista_unificado')['minutos_tocados']
    .sum()
    .reset_index()
    .sort_values('minutos_tocados', ascending=False)
    .head(15)
)

print(df_grafico2)

                    artista_unificado  minutos_tocados
25                            Liniker       896.417050
3                     Arnaldo Antunes       435.353817
12   Engenheiros e Humberto Gessinger       290.342650
30                         Nando Reis       246.314233
37                         Pink Floyd       108.556500
6                       Black Pantera        87.559500
38                              Pitty        64.581900
20                      Johnny Hooker        56.459800
14                      Gloria Groove        48.505933
31                     Ney Matogrosso        26.769217
50                        The Beatles        25.638050
43                     Punho de Mahin        25.388833
29                       Marisa Monte        24.398083
32  Nick Mason's Saucerful of Secrets        23.616667
35                      Pabllo Vittar        22.411083


In [34]:
# Minutos totais por artista (sem filtro de janela, só >= 2019)
df_totais = (
    df_com_data[df_com_data['ano_lancamento'] >= 2019]
    .groupby('artista_unificado')['minutos_tocados']
    .sum()
    .reset_index()
    .rename(columns={'minutos_tocados': 'minutos_totais'})
)

# Juntar com df_grafico2
df_scatter = df_grafico2.merge(df_totais, on='artista_unificado', how='left')

# Proporção: % dos minutos totais ouvidos nos primeiros 30 dias
df_scatter['proporcao'] = df_scatter['minutos_tocados'] / df_scatter['minutos_totais'] * 100

print(df_scatter[['artista_unificado', 'minutos_tocados', 'minutos_totais', 'proporcao']]
      .sort_values('proporcao', ascending=False))

                    artista_unificado  minutos_tocados  minutos_totais  \
8                       Gloria Groove        48.505933       57.220483   
6                               Pitty        64.581900       91.183083   
4                          Pink Floyd       108.556500      158.582600   
14                      Pabllo Vittar        22.411083       38.607650   
0                             Liniker       896.417050     2087.930183   
3                          Nando Reis       246.314233      704.182850   
10                        The Beatles        25.638050       74.097000   
9                      Ney Matogrosso        26.769217       77.959183   
1                     Arnaldo Antunes       435.353817     1367.989950   
11                     Punho de Mahin        25.388833       88.911817   
2    Engenheiros e Humberto Gessinger       290.342650     1202.782900   
13  Nick Mason's Saucerful of Secrets        23.616667      116.910533   
7                       Johnny Hooker 

In [35]:
df_scatter_filtrado = df_scatter[
    df_scatter['minutos_totais'] >= 50
].copy()

print(df_scatter_filtrado[['artista_unificado', 'minutos_tocados', 'minutos_totais', 'proporcao']]
      .sort_values('proporcao', ascending=False))

                    artista_unificado  minutos_tocados  minutos_totais  \
8                       Gloria Groove        48.505933       57.220483   
6                               Pitty        64.581900       91.183083   
4                          Pink Floyd       108.556500      158.582600   
0                             Liniker       896.417050     2087.930183   
3                          Nando Reis       246.314233      704.182850   
10                        The Beatles        25.638050       74.097000   
9                      Ney Matogrosso        26.769217       77.959183   
1                     Arnaldo Antunes       435.353817     1367.989950   
11                     Punho de Mahin        25.388833       88.911817   
2    Engenheiros e Humberto Gessinger       290.342650     1202.782900   
13  Nick Mason's Saucerful of Secrets        23.616667      116.910533   
7                       Johnny Hooker        56.459800      608.826650   
5                       Black Pantera 

In [36]:
fig_empolgacao = go.Figure()

# Calculate median proportion for dynamic text positioning
median_proporcao = df_scatter_filtrado['proporcao'].median()
text_positions = [
    'bottom center' if p > median_proporcao else 'top center'
    for p in df_scatter_filtrado['proporcao']
]

fig_empolgacao.add_trace(go.Scatter(
    x=df_scatter_filtrado['minutos_totais'],
    y=df_scatter_filtrado['proporcao'],
    mode='markers+text',
    cliponaxis=False, # Impede que o texto seja cortado na borda
    marker=dict(
        size=df_scatter_filtrado['minutos_tocados'] / df_scatter_filtrado['minutos_tocados'].max() * 60 + 10,
        color=df_scatter_filtrado['proporcao'],
        colorscale='Greens',
        showscale=True,
        colorbar=dict(title='% nos<br>30 dias'),
    ),
    text=df_scatter_filtrado['artista_unificado'],
    textposition=text_positions, # Use the dynamically generated positions
    customdata=list(zip(
        df_scatter_filtrado['artista_unificado'],
        df_scatter_filtrado['minutos_tocados'],
        df_scatter_filtrado['proporcao'],
    )),
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Minutos nos 30 dias: %{customdata[1]:.0f}<br>'
        'Proporção: %{customdata[2]:.1f}%<br>'
        'Minutos totais: %{x:.0f}'
        '<extra></extra>'
    ),
))

fig_empolgacao.update_layout(
    title=dict(
        text='Empolgação no lançamento<br><sup>Tamanho = minutos ouvidos nos primeiros 30 dias · Cor = proporção do total</sup>',
        x=0.5
    ),
    xaxis_title='Minutos totais ouvidos',
    yaxis_title='% ouvido nos primeiros 30 dias',
    yaxis=dict(ticksuffix='%'),
    margin=dict(l=80, r=80) # Adiciona um respiro nas laterais para textos longos
)

fig_empolgacao.show()


#3 - Pipeline do projeto

In [37]:
from IPython.display import HTML, display

pipeline_visao_geral = """
<div style="max-width: 800px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 720" xmlns="http://www.w3.org/2000/svg" style="background:#121212; border-radius:12px; padding:8px">
<defs>
  <marker id="arr1" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="#535353" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
</defs>

<!-- Estilos inline — paleta -->
<!-- Cinza neutro:  fill=#1E1E1E  stroke=#535353  text=#B3B3B3  -->
<!-- Verde Spotify: fill=#1A3D2B  stroke=#1DB954  text=#1DB954  -->
<!-- Roxo:          fill=#2A1F3D  stroke=#7B5EA7  text=#A67FD0  -->
<!-- Laranja:       fill=#3D2A1A  stroke=#E8821A  text=#E8821A  -->
<!-- Coral:         fill=#3D1F1F  stroke=#E85D5D  text=#E85D5D  -->

<!-- FONTE DE DADOS -->
<rect x="160" y="20" width="360" height="52" rx="8" fill="#1E1E1E" stroke="#535353" stroke-width="0.5"/>
<text x="340" y="40" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="14" font-weight="500" fill="#E0E0E0">Dados brutos do Spotify</text>
<text x="340" y="58" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="12" fill="#B3B3B3">31.515 execuções · JSON exportado</text>

<line x1="260" y1="72" x2="160" y2="108" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="420" y1="72" x2="520" y2="108" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>

<!-- NB1 -->
<rect x="40" y="108" width="280" height="72" rx="8" fill="#1A3D2B" stroke="#1DB954" stroke-width="0.5"/>
<text x="180" y="128" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#1DB954">NB1 — Engenharia de features</text>
<text x="180" y="146" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">Unificação de nomes · sessão · posição</text>
<text x="180" y="162" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">lealdade · longevidade · período do dia</text>

<!-- NB2 -->
<rect x="360" y="108" width="280" height="72" rx="8" fill="#1A3D2B" stroke="#1DB954" stroke-width="0.5"/>
<text x="500" y="128" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#1DB954">NB2 — API Spotify</text>
<text x="500" y="146" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">duração real · data de lançamento</text>
<text x="500" y="162" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">600 chamadas/dia · Todas URIs coletadas</text>

<line x1="180" y1="180" x2="260" y2="230" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="500" y1="180" x2="420" y2="230" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>

<!-- NB3 -->
<rect x="140" y="230" width="400" height="72" rx="8" fill="#1A3D2B" stroke="#1DB954" stroke-width="0.5"/>
<text x="340" y="250" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#1DB954">NB3 — Features de execução</text>
<text x="340" y="268" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">retenção · escuta completa · skip · início intencional</text>
<text x="340" y="284" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">duração estimada · fonte de duração</text>

<line x1="340" y1="302" x2="340" y2="330" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>

<!-- NB4 -->
<rect x="100" y="330" width="480" height="100" rx="8" fill="#2A1F3D" stroke="#7B5EA7" stroke-width="0.5"/>
<text x="340" y="354" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#A67FD0">NB4 — Modelagem</text>
<text x="340" y="374" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">PCA (8 features → 5 componentes)</text>
<text x="340" y="390" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">GMM (6 clusters) · Item2Vec (64 dim, janela 5)</text>
<text x="340" y="406" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">Markov (transições de sessão) · score = freq × retenção</text>

<line x1="200" y1="430" x2="130" y2="478" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="340" y1="430" x2="340" y2="478" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="480" y1="430" x2="550" y2="478" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>

<!-- TRÊS PLAYLISTS -->
<rect x="40" y="478" width="180" height="88" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="130" y="500" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Início do dia</text>
<text x="130" y="518" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">50 músicas</text>
<text x="130" y="534" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 0, 3, 4</text>
<text x="130" y="550" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">posição sessão ≤ 0.4</text>

<rect x="250" y="478" width="180" height="88" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="340" y="500" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Viagem longa</text>
<text x="340" y="518" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">100 músicas</text>
<text x="340" y="534" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 0, 1, 3, 4, 5</text>
<text x="340" y="550" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">retenção ≥ 0.6</text>

<rect x="460" y="478" width="180" height="88" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="550" y="500" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Fim do dia</text>
<text x="550" y="518" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">50 músicas</text>
<text x="550" y="534" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 3, 4, 5</text>
<text x="550" y="550" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">posição sessão ≥ 0.8</text>

<line x1="130" y1="566" x2="210" y2="610" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="340" y1="566" x2="340" y2="610" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>
<line x1="550" y1="566" x2="470" y2="610" stroke="#535353" stroke-width="1" marker-end="url(#arr1)"/>

<!-- NB5 -->
<rect x="140" y="610" width="400" height="52" rx="8" fill="#3D2A1A" stroke="#E8821A" stroke-width="0.5"/>
<text x="340" y="630" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E8821A">NB5 — Dashboards e apresentação</text>
<text x="340" y="648" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8B87A">Passado · Presente · Futuro</text>

</svg>
</div>
"""

display(HTML(pipeline_visao_geral))

In [40]:
pipeline_nb4 = """
<div style="max-width: 800px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 680" xmlns="http://www.w3.org/2000/svg" style="background:#121212; border-radius:12px; padding:8px">
<defs>
  <marker id="arr2" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="#535353" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
</defs>

<!-- ENTRADA -->
<rect x="160" y="20" width="360" height="44" rx="8" fill="#1E1E1E" stroke="#535353" stroke-width="0.5"/>
<text x="340" y="38" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="14" font-weight="500" fill="#E0E0E0">df_ranking — 4.215 músicas únicas</text>
<text x="340" y="56" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="12" fill="#B3B3B3">3.010 com embedding · 1.205 sem embedding</text>

<line x1="260" y1="64" x2="160" y2="112" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>
<line x1="420" y1="64" x2="520" y2="112" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>

<!-- PCA -->
<rect x="40" y="112" width="280" height="88" rx="8" fill="#1A2A3D" stroke="#4A90D9" stroke-width="0.5"/>
<text x="180" y="134" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#7BB8F0">PCA — 5 componentes</text>
<text x="180" y="152" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">retenção · escuta completa · skip</text>
<text x="180" y="168" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">início intencional · duração estimada</text>
<text x="180" y="184" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">lealdade · longevidade · total plays</text>

<line x1="180" y1="200" x2="180" y2="228" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>

<!-- GMM -->
<rect x="40" y="228" width="280" height="136" rx="8" fill="#1A2A3D" stroke="#4A90D9" stroke-width="0.5"/>
<text x="180" y="248" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#7BB8F0">GMM — 6 clusters</text>
<text x="180" y="264" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">0 · Exploração de artistas favoritos</text>
<text x="180" y="278" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">1 · Indiferença passiva</text>
<text x="180" y="292" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">2 · Tentativas que não colam</text>
<text x="180" y="306" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">3 · Longas e intencionais</text>
<text x="180" y="320" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">4 · Depende do momento</text>
<text x="180" y="334" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#A8CFF5">5 · Hits Pessoais</text>

<!-- Item2Vec -->
<rect x="360" y="112" width="280" height="104" rx="8" fill="#2A1F3D" stroke="#7B5EA7" stroke-width="0.5"/>
<text x="500" y="134" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#A67FD0">Item2Vec</text>
<text x="500" y="152" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">token = música — artista</text>
<text x="500" y="168" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">janela 5 · 64 dimensões · 10 épocas</text>
<text x="500" y="184" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">min_count 2 · skip-gram</text>
<text x="500" y="200" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">vocabulário: 3.010 tokens</text>

<line x1="500" y1="216" x2="500" y2="254" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>

<!-- Markov -->
<rect x="360" y="254" width="280" height="72" rx="8" fill="#2A1F3D" stroke="#7B5EA7" stroke-width="0.5"/>
<text x="500" y="266" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#A67FD0">Cadeia de Markov</text>
<text x="500" y="284" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">transições entre tokens por sessão</text>
<text x="500" y="300" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#C4A8E8">ordenação global + local por bloco</text>

<line x1="180" y1="364" x2="240" y2="384" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>
<line x1="500" y1="326" x2="440" y2="384" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>

<!-- montar_playlist -->
<rect x="100" y="384" width="480" height="88" rx="8" fill="#1A3D2B" stroke="#1DB954" stroke-width="0.5"/>
<text x="340" y="406" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#1DB954">montar_playlist()</text>
<text x="340" y="424" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">pool = clusters + embedding disponível</text>
<text x="340" y="440" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">score = freq_contexto × retenção_média</text>
<text x="340" y="456" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#86C9A0">limite por artista → Markov global → Markov local</text>

<line x1="200" y1="472" x2="120" y2="530" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>
<line x1="340" y1="472" x2="340" y2="530" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>
<line x1="480" y1="472" x2="560" y2="530" stroke="#535353" stroke-width="1" marker-end="url(#arr2)"/>

<!-- Três playlists -->
<rect x="20" y="530" width="200" height="100" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="120" y="552" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Início do dia</text>
<text x="120" y="570" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">50 músicas · lim. 3/artista</text>
<text x="120" y="586" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 0, 3, 4</text>
<text x="120" y="602" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">posição sessão ≤ 0.4</text>
<text x="120" y="618" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">plays ≥ 3 · freq ≥ 1</text>

<rect x="240" y="530" width="200" height="100" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="340" y="552" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Viagem longa</text>
<text x="340" y="570" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">100 músicas · lim. 6/artista</text>
<text x="340" y="586" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 0, 1, 3, 4, 5</text>
<text x="340" y="602" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">retenção ≥ 0.6</text>
<text x="340" y="618" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">sem filtro de posição</text>

<rect x="460" y="530" width="200" height="100" rx="8" fill="#3D1F1F" stroke="#E85D5D" stroke-width="0.5"/>
<text x="560" y="552" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="13" font-weight="500" fill="#E85D5D">Fim do dia</text>
<text x="560" y="570" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">50 músicas · lim. 3/artista</text>
<text x="560" y="586" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">clusters 3, 4, 5</text>
<text x="560" y="602" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">posição sessão ≥ 0.8</text>
<text x="560" y="618" text-anchor="middle" dominant-baseline="central" font-family="monospace" font-size="11" fill="#E8A0A0">sem filtro extra</text>

</svg>
</div>
"""

display(HTML(pipeline_nb4))

#4. Gráficos por Clusters


In [39]:
from sklearn.decomposition import PCA

# Colunas de embedding
colunas_emb = [f'emb_{i}' for i in range(64)]

# Apenas músicas com embedding
df_emb = df_ranking[df_ranking['emb_0'].notna()].copy()

# PCA 2D — só para visualização
pca_vis = PCA(n_components=2, random_state=42)
coords = pca_vis.fit_transform(df_emb[colunas_emb])

df_emb['pca_x'] = coords[:, 0]
df_emb['pca_y'] = coords[:, 1]

variancia = pca_vis.explained_variance_ratio_
print(f"Variância explicada — PC1: {variancia[0]:.1%}  PC2: {variancia[1]:.1%}  Total: {sum(variancia):.1%}")

Variância explicada — PC1: 34.9%  PC2: 21.1%  Total: 56.0%


In [41]:
# === ENRIQUECIMENTO DO DF_RANKING COM METADADOS DE CLUSTER ===

# Merge com cluster_meta para trazer nome oficial
df_ranking = df_ranking.merge(
    cluster_meta[['cluster', 'cluster_nome']],
    on='cluster',
    how='left'
)

# Validação
nulos = df_ranking['cluster_nome'].isna().sum()
print(f"Registros sem nome de cluster: {nulos}")

# Cores e descrições sincronizadas com a ordem do arquivo meta
lista_clusters_oficial = cluster_meta.sort_values('cluster')['cluster_nome'].tolist()
cores_clusters = {
    nome: paleta_cores[i % len(paleta_cores)]
    for i, nome in enumerate(lista_clusters_oficial)
}
descricoes = dict(zip(cluster_meta['cluster_nome'], cluster_meta['cluster_descricao']))

print(f"\nCores mapeadas para {len(cores_clusters)} clusters:")
for nome, cor in cores_clusters.items():
    print(f"  {nome}: {cor}")

Registros sem nome de cluster: 0

Cores mapeadas para 6 clusters:
  Exploração de artistas favoritos: #DA77DA
  Indiferença passiva: #C8A800
  Tentativas que não colam: #2ECC71
  Longas e intencionais: #00BCD4
  Depende do momento: #748FFC
  Hits Pessoais: #FF6B6B


In [42]:
# 1. Recriar df_emb garantindo que ele herde as colunas novas e as coordenadas do PCA
df_emb = df_ranking[df_ranking['emb_0'].notna()].copy()

# Como coords foi gerado na ordem do df_emb original, resetamos o index para garantir o alinhamento
# Ou simplesmente atribuímos, já que df_emb e coords têm o mesmo tamanho (3010)
df_emb['pca_x'] = coords[:, 0]
df_emb['pca_y'] = coords[:, 1]

# 2. Mapa PCA sincronizado com cores_clusters
fig_pca = go.Figure()

# Itera sobre os nomes de clusters únicos presentes nos dados com embedding
for cluster_nome in df_emb['cluster_nome'].unique():
    df_c = df_emb[df_emb['cluster_nome'] == cluster_nome]

    # Busca a cor no mapeamento global centralizado
    cor = cores_clusters.get(cluster_nome, '#535353')

    fig_pca.add_trace(go.Scatter(
        x=df_c['pca_x'],
        y=df_c['pca_y'],
        mode='markers',
        name=cluster_nome,
        marker=dict(
            color=cor,
            size=5,
            opacity=0.6,
        ),
        customdata=df_c[['musica_unificada', 'artista_unificado', 'cluster_nome']].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "%{customdata[1]}<br>"
            "<i>%{customdata[2]}</i>"
            "<extra></extra>"
        ),
    ))

fig_pca.update_layout(
    title=f"Mapa de músicas — PCA dos embeddings Item2Vec<br>"
          f"<sup>Variância explicada: PC1 {variancia[0]:.1%} · PC2 {variancia[1]:.1%} · Total {sum(variancia):.1%}</sup>",
    xaxis=dict(title="Componente principal 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Componente principal 2", showgrid=False, zeroline=False),
    height=550,
    legend=dict(orientation="v", x=1.02, y=1, xanchor="left"),
)

fig_pca.show()

In [43]:
from sklearn.preprocessing import MinMaxScaler

features_radar = [
    'retencao_media',
    'escuta_completa_rate',
    'skip_rate',
    'inicio_intencional_rate',
    'duracao_estimada_ms',
    'rating_lealdade_artista',
    'longevidade_dias_artista',
    'total_plays',
]

nomes_radar = [
    'Retenção',
    'Escuta completa',
    'Skip rate',
    'Início intencional',
    'Duração',
    'Lealdade do artista',
    'Longevidade do artista',
    'Total de plays',
]

# Média por cluster
perfil_clusters = (
    df_ranking
    .groupby('cluster_nome')[features_radar]
    .mean()
    .reset_index()
)

# Normalizar para 0-1
scaler = MinMaxScaler()
perfil_normalizado = perfil_clusters.copy()
perfil_normalizado[features_radar] = scaler.fit_transform(perfil_clusters[features_radar])

print(perfil_normalizado[['cluster_nome'] + features_radar].round(2).to_string())

                       cluster_nome  retencao_media  escuta_completa_rate  skip_rate  inicio_intencional_rate  duracao_estimada_ms  rating_lealdade_artista  longevidade_dias_artista  total_plays
0                Depende do momento            0.08                  0.14       1.00                     0.19                 0.16                     0.65                      0.92         0.21
1  Exploração de artistas favoritos            0.16                  0.13       0.39                     0.92                 0.07                     0.60                      1.00         0.07
2                     Hits Pessoais            0.39                  0.41       0.49                     0.30                 0.00                     1.00                      0.48         1.00
3               Indiferença passiva            1.00                  1.00       0.00                     0.00                 0.07                     0.46                      0.74         0.00
4             Longas e in

In [44]:
# Gráfico de Radar sincronizado com cores_clusters
fig_radar = go.Figure()

for _, row in perfil_normalizado.iterrows():
    nome = row['cluster_nome']
    valores = row[features_radar].tolist()
    valores_fechados = valores + [valores[0]]
    eixos_fechados = nomes_radar + [nomes_radar[0]]

    # Busca a cor no mapeamento global centralizado
    cor_cluster = cores_clusters.get(nome, '#535353')

    fig_radar.add_trace(go.Scatterpolar(
        r=valores_fechados,
        theta=eixos_fechados,
        fill='toself',
        fillcolor=cor_cluster,
        line=dict(color=cor_cluster, width=1.5),
        opacity=0.25,
        name=nome,
        hovertemplate="<b>" + nome + "</b><br>%{theta}: %{r:.2f}<extra></extra>",
    ))

    # Linha do contorno
    fig_radar.add_trace(go.Scatterpolar(
        r=valores_fechados,
        theta=eixos_fechados,
        mode='lines',
        line=dict(color=cor_cluster, width=2),
        opacity=0.9,
        name=nome,
        showlegend=False,
        hoverinfo='skip',
    ))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1], tickfont=dict(size=10), gridcolor='#333333'),
        angularaxis=dict(tickfont=dict(size=11), gridcolor='#333333'),
        bgcolor='#1A1A1A',
    ),
    title="Perfil dos clusters — features normalizadas 0–1",
    showlegend=True,
    legend=dict(orientation="v", x=1.1, y=1),
    height=550,
)

fig_radar.show()

In [45]:
# Top 5 músicas por cluster
top_por_cluster = (
    df_ranking
    .groupby('cluster_nome', group_keys=False)
    .apply(lambda x: x.nlargest(5, 'total_plays'))
    .reset_index(drop=True)
)

print(f"Clusters identificados no novo Top 5: {top_por_cluster['cluster_nome'].unique().tolist()}")

Clusters identificados no novo Top 5: ['Depende do momento', 'Exploração de artistas favoritos', 'Hits Pessoais', 'Indiferença passiva', 'Longas e intencionais', 'Tentativas que não colam']


In [46]:
# === Configuração Central de Identidade Visual ===
# Sincronização dinâmica de cores e descrições baseada no cluster_meta

lista_clusters_oficial = cluster_meta.sort_values('cluster')['cluster_nome'].tolist()

cores_clusters = {
    nome: paleta_cores[i % len(paleta_cores)]
    for i, nome in enumerate(lista_clusters_oficial)
}

descricoes = dict(zip(cluster_meta['cluster_nome'], cluster_meta['cluster_descricao']))

print(f"Identidade visual pronta para {len(cores_clusters)} clusters.")

Identidade visual pronta para 6 clusters.


In [47]:
from IPython.display import HTML, display

def gerar_cards_clusters(df_top, descricoes, cores):
    html = """
    <div style="
        background:#121212;
        padding:24px;
        border-radius:12px;
        display:grid;
        grid-template-columns: repeat(3, 1fr);
        gap:16px;
        font-family:monospace;
    ">
    """

    # Usamos a ordem baseada no cluster_meta para manter consistência
    ordem = cluster_meta.sort_values('cluster')['cluster_nome'].unique().tolist()

    for nome in ordem:
        if nome not in df_top['cluster_nome'].values: continue

        cor = cores.get(nome, '#535353')
        desc = descricoes.get(nome, '---')
        musicas = df_top[df_top['cluster_nome'] == nome]

        linhas_musicas = ""
        for _, m in musicas.iterrows():
            linhas_musicas += f"""
            <div style="
                padding:6px 8px;
                border-radius:4px;
                background:#1E1E1E;
                margin-bottom:4px;
            ">
                <div style="color:#E0E0E0; font-size:12px; font-weight:500;">
                    {m['musica_unificada']}
                </div>
                <div style="color:#B3B3B3; font-size:11px;">
                    {m['artista_unificado']}
                </div>
            </div>
            """

        html += f"""
        <div style="
            background:#1A1A1A;
            border:1px solid {cor}40;
            border-top:3px solid {cor};
            border-radius:8px;
            padding:16px;
        ">
            <div style="color:{cor}; font-size:14px; font-weight:700; margin-bottom:6px;">
                {nome}
            </div>
            <div style="color:#888888; font-size:11px; margin-bottom:12px; line-height:1.5;">
                {desc}
            </div>
            <div style="font-size:11px; color:#535353; margin-bottom:8px; text-transform:uppercase; letter-spacing:1px;">
                Top músicas
            </div>
            {linhas_musicas}
        </div>
        """

    html += "</div>"
    return html

display(HTML(gerar_cards_clusters(top_por_cluster, descricoes, cores_clusters)))

# 6. Gráficos por playlists

In [48]:
# Join para trazer nome do cluster e métricas completas
colunas_ranking = ['musica_unificada', 'artista_unificado', 'skip_rate',
                   'inicio_intencional_rate', 'cluster_nome']

def enriquecer_playlist(pl, nome):
    df = pl.drop(columns=['Unnamed: 0'], errors='ignore').copy()
    df = df.merge(
        df_ranking[colunas_ranking],
        on=['musica_unificada', 'artista_unificado'],
        how='left'
    )
    df['playlist'] = nome
    return df

pl_i = enriquecer_playlist(pl_inicio_dia, 'Início do dia')
pl_f = enriquecer_playlist(pl_fim_dia, 'Fim do dia')
pl_v = enriquecer_playlist(pl_viagem, 'Viagem longa')

# Conferir
for pl, nome in [(pl_i, 'Início'), (pl_f, 'Fim'), (pl_v, 'Viagem')]:
    print(f"{nome}: {len(pl)} músicas · {pl['cluster_nome'].isna().sum()} sem cluster_nome")

Início: 50 músicas · 0 sem cluster_nome
Fim: 50 músicas · 0 sem cluster_nome
Viagem: 100 músicas · 0 sem cluster_nome


In [49]:
pl_todas = pd.concat([pl_i, pl_f, pl_v], ignore_index=True)

dist_clusters = (
    pl_todas
    .groupby(['playlist', 'cluster_nome'])
    .size()
    .reset_index(name='count')
)

ordem_playlists = ['Início do dia', 'Fim do dia', 'Viagem longa']

fig_dist = go.Figure()

for cluster_nome, cor in cores_clusters.items():
    df_c = dist_clusters[dist_clusters['cluster_nome'] == cluster_nome]
    fig_dist.add_trace(go.Bar(
        x=df_c['playlist'],
        y=df_c['count'],
        name=cluster_nome,
        marker_color=cor,
        hovertemplate="%{x}<br>" + cluster_nome + ": %{y} músicas<extra></extra>",
    ))

fig_dist.update_layout(
    barmode='stack',
    title='Composição das playlists por cluster',
    xaxis=dict(
        categoryorder='array',
        categoryarray=ordem_playlists,
        title='',
    ),
    yaxis=dict(title='Músicas'),
    height=420,
    legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'),
)

fig_dist.show()

In [50]:
metricas = (
    pl_todas
    .groupby('playlist')[['retencao_media', 'skip_rate', 'inicio_intencional_rate']]
    .mean()
    .reset_index()
    .round(2)
)

fig_metricas = go.Figure()

config_metricas = [
    ('retencao_media',           'Retenção média',      '#1DB954'),
    ('skip_rate',                'Skip rate',           '#E85D5D'),
    ('inicio_intencional_rate',  'Início intencional',  '#4A90D9'),
]

for coluna, label, cor in config_metricas:
    fig_metricas.add_trace(go.Bar(
        x=metricas['playlist'],
        y=metricas[coluna],
        name=label,
        marker_color=cor,
        text=metricas[coluna],
        textposition='outside',
        textfont=dict(size=11),
        hovertemplate="%{x}<br>" + label + ": %{y:.2f}<extra></extra>",
    ))

fig_metricas.update_layout(
    barmode='group',
    title='Métricas médias por playlist',
    xaxis=dict(
        categoryorder='array',
        categoryarray=ordem_playlists,
        title='',
    ),
    yaxis=dict(
        title='',
        range=[0, 1.15],
    ),
    height=420,
    legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'),
)

fig_metricas.show()

In [51]:
artistas = (
    pl_todas
    .groupby('playlist')
    .agg(
        artistas_unicos=('artista_unificado', 'nunique'),
        total_musicas=('musica_unificada', 'count'),
        minutos_totais=('duracao_estimada_ms', lambda x: (x.sum() / 60000).round(0)),
    )
    .reset_index()
)

fig_artistas = go.Figure()

fig_artistas.add_trace(go.Bar(
    x=artistas['playlist'],
    y=artistas['artistas_unicos'],
    name='Artistas únicos',
    marker_color='#1DB954',
    text=artistas['artistas_unicos'],
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate="%{x}<br>%{y} artistas únicos<extra></extra>",
))

fig_artistas.update_layout(
    title='Artistas únicos por playlist',
    xaxis=dict(
        categoryorder='array',
        categoryarray=ordem_playlists,
        title='',
    ),
    yaxis=dict(
        title='Artistas únicos',
        range=[0, artistas['artistas_unicos'].max() * 1.2],
    ),
    height=380,
)

fig_artistas.show()

# Imprimir também os minutos totais estimados
for _, row in artistas.iterrows():
    horas = int(row['minutos_totais'] // 60)
    mins = int(row['minutos_totais'] % 60)
    print(f"{row['playlist']}: {row['artistas_unicos']} artistas · {row['total_musicas']} músicas · ~{horas}h{mins:02d}min")

Fim do dia: 28 artistas · 50 músicas · ~3h22min
Início do dia: 28 artistas · 50 músicas · ~3h52min
Viagem longa: 38 artistas · 100 músicas · ~6h30min


In [52]:
def formatar_duracao(ms):
    if pd.isna(ms) or ms == 0:
        return "--"
    total_seg = int(ms / 1000)
    return f"{total_seg // 60}:{total_seg % 60:02d}"

def gerar_playlist_html(df_pl, titulo, cor):
    html = f"""
    <div style="
        background:#121212;
        padding:24px;
        border-radius:12px;
        font-family:monospace;
        margin-bottom:32px;
    ">
        <div style="
            border-left:4px solid {cor};
            padding-left:16px;
            margin-bottom:20px;
        ">
            <div style="color:{cor}; font-size:20px; font-weight:700;">{titulo}</div>
            <div style="color:#B3B3B3; font-size:12px; margin-top:4px;">
                {len(df_pl)} músicas · {df_pl['artista_unificado'].nunique()} artistas
            </div>
        </div>
        <div style="
            display:grid;
            grid-template-columns: repeat(2, 1fr);
            gap:8px;
        ">
    """

    for i, (_, row) in enumerate(df_pl.iterrows(), 1):
        cluster_nome = row.get('cluster_nome', '—')
        cor_cluster = cores_clusters.get(cluster_nome, '#535353')
        duracao = formatar_duracao(row.get('duracao_estimada_ms', 0))
        retencao = f"{row['retencao_media']:.0%}" if pd.notna(row.get('retencao_media')) else '—'

        html += f"""
        <div style="
            background:#1A1A1A;
            border:1px solid #2A2A2A;
            border-radius:6px;
            padding:10px 12px;
            display:flex;
            align-items:center;
            gap:12px;
        ">
            <div style="
                color:#535353;
                font-size:11px;
                min-width:24px;
                text-align:right;
            ">{i}</div>
            <div style="flex:1; min-width:0;">
                <div style="
                    color:#E0E0E0;
                    font-size:12px;
                    font-weight:500;
                    white-space:nowrap;
                    overflow:hidden;
                    text-overflow:ellipsis;
                ">{row['musica_unificada']}</div>
                <div style="
                    color:#B3B3B3;
                    font-size:11px;
                    white-space:nowrap;
                    overflow:hidden;
                    text-overflow:ellipsis;
                ">{row['artista_unificado']}</div>
            </div>
            <div style="display:flex; flex-direction:column; align-items:flex-end; gap:3px; flex-shrink:0;">
                <div style="
                    background:{cor_cluster}22;
                    color:{cor_cluster};
                    font-size:10px;
                    padding:2px 6px;
                    border-radius:3px;
                    white-space:nowrap;
                ">{cluster_nome}</div>
                <div style="display:flex; gap:8px;">
                    <span style="color:#535353; font-size:10px;">↺ {retencao}</span>
                    <span style="color:#535353; font-size:10px;">⏱ {duracao}</span>
                </div>
            </div>
        </div>
        """

    html += "</div></div>"
    return html

In [53]:
display(HTML(gerar_playlist_html(pl_i, "Início do dia", "#1DB954")))
display(HTML(gerar_playlist_html(pl_f, "Fim do dia",    "#4A90D9")))
display(HTML(gerar_playlist_html(pl_v, "Viagem longa",  "#E8821A")))

# 9 - Geração e Exportação do HTML

In [55]:
from plotly.io import to_html
from IPython.display import HTML

# Monta o HTML completo
secoes = []

# Adicionar o sumário navegável no início com nova hierarquia
sumario_html = """
<div style="
    background:#121212;
    padding:24px;
    border-radius:12px;
    font-family:monospace;
    margin-bottom:32px;
    max-width: 800px; margin: 0 auto;
">
    <h3 style="color:#1DB954; margin-top:0; margin-bottom:16px;">Sumário</h3>
    <ul style="list-style:none; padding:0;">
        <li style="margin-bottom:8px;"><a href="#passado-analise-exploratoria-temporal" style="color:#1DB954; text-decoration:none; font-weight:bold;">Passado: Análise Exploratória e Temporal</a>
            <ul style="list-style:none; padding-left: 20px; margin-top: 4px;">
                <li style="margin-bottom:8px;"><a href="#retrospectivas-anuais" style="color:#B3B3B3; text-decoration:none;">Retrospectivas Anuais</a>
                </li>
                <li style="margin-bottom:8px;"><a href="#analise-por-tempo-e-sessao" style="color:#B3B3B3; text-decoration:none;">Análise por Tempo e Sessão</a>
                </li>
                <li style="margin-bottom:8px;"><a href="#analise-por-lancamento" style="color:#B3B3B3; text-decoration:none;">Análise por Data de Lançamento</a></li>
            </ul>
        </li>
        <li style="margin-bottom:8px;"><a href="#presente-modelagem-recomendacoes" style="color:#1DB954; text-decoration:none; font-weight:bold;">Presente: Modelagem e Recomendações</a>
            <ul style="list-style:none; padding-left: 20px; margin-top: 4px;">
                <li style="margin-bottom:8px;"><a href="#pipeline-do-projeto" style="color:#B3B3B3; text-decoration:none;">Pipeline do Projeto</a></li>
                <li style="margin-bottom:8px;"><a href="#detalhes-do-modelo" style="color:#B3B3B3; text-decoration:none;">Detalhes do Modelo (NB4)</a></li>
                <li style="margin-bottom:8px;"><a href="#clusters-e-similaridade" style="color:#B3B3B3; text-decoration:none;">Clusters e Similaridade Musical</a></li>
                <li style="margin-bottom:8px;"><a href="#cards-dos-clusters" style="color:#B3B3B3; text-decoration:none;">Cards dos Clusters</a></li>

            </ul>
        </li>
        <li style="margin-bottom:8px;"><a href="#futuro-playlists" style="color:#1DB954; text-decoration:none; font-weight:bold;">Futuro: Playlists a Serem Curtidas</a>
            <ul style="list-style:none; padding-left: 20px; margin-top: 4px;">
                <li style="margin-bottom:8px;"><a href="#playlists-geradas" style="color:#B3B3B3; text-decoration:none;">Playlists Geradas</a></li>
                <li style="margin-bottom:8px;"><a href="#estatisticas-playlist" style="color:#B3B3B3; text-decoration:none;">Estatísticas das Playlists</a></li>
            </ul>
        </li>
    </ul>
</div>
"""
#secoes.append(sumario_html)


# -- Análise Exploratória e Temporal --
secoes.append('<h2 id="passado-analise-exploratoria-temporal" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Passado: Análise Exploratória e Temporal</h2>')

# Retrospectivas Anuais (antigo h2, agora h3)
secoes.append('<h3 id="retrospectivas-anuais" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Retrospectivas Anuais</h3>')

# Plotly — inclui a lib só na primeira fig
secoes.append(fig_retro.to_html(full_html=False, include_plotlyjs=True))
secoes.append(fig_minutos.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_bump.to_html(full_html=False, include_plotlyjs=False))


# Análise por Sessão e Tempo (antigo h2, agora h3)
secoes.append('<h3 id="analise-por-tempo-e-sessao" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Análise por Tempo e Sessão</h3>')

secoes.append(fig_mes.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_heatmap.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_periodo.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_sessoes.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_sessoes_ano.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_hora_inicio.to_html(full_html=False, include_plotlyjs=False))

# Análise por Data de Lançamento (antigo h2, agora h3)
secoes.append('<h3 id="analise-por-lancamento" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Análise por Data de Lançamento</h3>')
secoes.append(fig_lancamento.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_empolgacao.to_html(full_html=False, include_plotlyjs=False))

# -- Modelagem e Recomendações --
secoes.append('<h2 id="presente-modelagem-recomendacoes" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Presente: Modelagem e Recomendações</h2>')

# Pipeline do Projeto (antigo h2, agora h3)
secoes.append('<h3 id="pipeline-do-projeto" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Pipeline do Projeto</h3>')
secoes.append(pipeline_visao_geral)

# Detalhes do Modelo (NB4) (antigo h2, agora h3)
secoes.append('<h3 id="detalhes-do-modelo" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Detalhes do Modelo (NB4)</h3>')
secoes.append(pipeline_nb4)

# Clusters e Similaridade Musical (antigo h2, agora h3)
secoes.append('<h3 id="clusters-e-similaridade" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Clusters e Similaridade Musical</h3>')
secoes.append(fig_pca.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_radar.to_html(full_html=False, include_plotlyjs=False))

# Cards dos Clusters (antigo h2, agora h3)
secoes.append('<h3 id="cards-dos-clusters" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Cards dos Clusters</h3>')
secoes.append(gerar_cards_clusters(top_por_cluster, descricoes, cores_clusters))

# -- Playlists --
secoes.append('<h2 id="futuro-playlists" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Futuro: Playlistas a Serem Curtidas</h2>')


# Playlists Individuais (antigo h2, agora h3)
secoes.append('<h3 id="playlists-geradas" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Playlists Geradas</h3>')
secoes.append(gerar_playlist_html(pl_i, "Início do dia", "#1DB954"))
secoes.append(gerar_playlist_html(pl_f, "Fim do dia", "#4A90D9"))
secoes.append(gerar_playlist_html(pl_v, "Viagem longa", "#E8821A"))

# Playlists Geradas (antigo h2, agora h3)
secoes.append('<h3 id="estatisticas-playlists" style="color:#1DB954; font-family:monospace; text-align:center; padding: 20px 0 0 0;">Estatísticas das Playlists</h3>')
secoes.append(fig_dist.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_metricas.to_html(full_html=False, include_plotlyjs=False))
secoes.append(fig_artistas.to_html(full_html=False, include_plotlyjs=False))


html_header = """
<html>
<head>
<meta charset="utf-8">
<title>Spotify Curadoria — Rodney Takemi</title>
<style>
  body { background-color: #121212; margin: 0; padding: 16px; }
</style>
</head>
<body>
<div style="text-align:center; padding: 40px 16px 20px 16px; font-family: monospace;">
  <h1 style="color:#1DB954; font-size: 2em; margin: 0;">Spotify Curadoria</h1>
  <p style="color:#B3B3B3; margin: 8px 0 0 0;">7 anos de histórico musical · Rodney Takemi</p>
  <p style="color:#B3B3B3; max-width: 640px; margin: 16px auto 0 auto; font-size: 0.85em; line-height: 1.7;">
    Este projeto analisa 7 anos de histórico de escuta pessoal no Spotify — 28 mil execuções, 9 mil músicas únicas. O passado é explorado através de análises e visualizações interativas de comportamento musical. O presente é interpretado por técnicas de machine learning: Item2Vec, PCA, GMM e Cadeias de Markov. O futuro são três playlists personalizadas, geradas a partir de tudo isso.
  </p>
</div>
"""

html_footer = """
<div style="text-align:center; padding: 40px; font-family: monospace; color: #B3B3B3;">
  <p>Notebooks de referência:</p>
  <a href="https://nbviewer.org/github/rodneytakemi/spotify-curadoria/blob/main/notebooks/nb1.ipynb" style="color:#1DB954; margin: 0 12px;">NB1</a>
  <a href="https://nbviewer.org/github/rodneytakemi/spotify-curadoria/blob/main/notebooks/nb2.ipynb" style="color:#1DB954; margin: 0 12px;">NB2</a>
  <a href="https://nbviewer.org/github/rodneytakemi/spotify-curadoria/blob/main/notebooks/nb3.ipynb" style="color:#1DB954; margin: 0 12px;">NB3</a>
  <a href="https://nbviewer.org/github/rodneytakemi/spotify-curadoria/blob/main/notebooks/nb4.ipynb" style="color:#1DB954; margin: 0 12px;">NB4</a>
  <a href="https://nbviewer.org/github/rodneytakemi/spotify-curadoria/blob/main/notebooks/nb5.ipynb" style="color:#1DB954; margin: 0 12px;">NB5</a>
  <a href="https://github.com/rodneytakemi/spotify-curadoria" style="color:#1DB954; margin: 0 12px;">GitHub</a>
</div>
</body>
</html>
"""

with open('/content/index5.html', 'w', encoding='utf-8') as f:
    f.write(html_header)
    f.write('\n'.join(secoes))
    f.write(html_footer)

In [ ]:
from IPython.display import HTML

# Read the content of the generated HTML file
with open('/content/index5.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Display the HTML content directly in the notebook
display(HTML(html_content))

Output hidden; open in https://colab.research.google.com to view.